In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import re
import os
import json

from google.colab import files

uploaded = files.upload()

df = pd.read_csv("unified_safety 3.csv")
print("Dataset loaded successfully")
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Saving unified_safety 3.csv to unified_safety 3 (5).csv
Dataset loaded successfully
Shape: (2037, 29)

Columns:
['zone_id', 'latitude', 'longitude', 'wind_speed_ms', 'wave_height_m', 'wave_period_s', 'wave_direction_deg', 'saral_sst_c', 'saral_distance_km', 'saral_source', 'current_u_ms', 'current_v_ms', 'current_speed_ms', 'cyclone_distance_km', 'cyclone_wind_kt', 'cyclone_grade', 'cyclone_pressure_hpa', 'cyclone_pressure_drop_hpa', 'cyclone_basin', 'rainfall_mean', 'rainfall_max', 'risk_score', 'risk_level', 'agent', 'data_region', 'region', 'dist_coast_m', 'ssha_m', 'saral_time']


In [ ]:
print("Number of rows:", len(df))
print("Number of columns:", len(df.columns))

print("\nRegions:")
print(df["region"].value_counts())

print("\nMissing values:")
print(df.isna().sum().sort_values(ascending=False))

print("\nFirst 5 rows:")
display(df.head())

Number of rows: 2037
Number of columns: 29

Regions:
region
INDIAN_MARINE          1700
KERALA                  110
MAHARASHTRA              80
NORTH TAMILNADU          60
SOUTH ANDHRAPRADESH      23
ANDAMAN                  20
KARNATAKA                20
LAKSHADWEEP              17
GOA                       7
Name: count, dtype: int64

Missing values:
saral_time                   1995
dist_coast_m                 1995
ssha_m                       1995
cyclone_pressure_hpa         1256
rainfall_mean                 702
rainfall_max                  702
saral_sst_c                   602
wave_height_m                 598
wave_period_s                 594
wave_direction_deg            594
cyclone_grade                 564
current_speed_ms              495
current_u_ms                  495
current_v_ms                  495
cyclone_pressure_drop_hpa     241
cyclone_wind_kt               241
cyclone_basin                 241
wind_speed_ms                 150
zone_id                         0

,zone_id,latitude,longitude,wind_speed_ms,wave_height_m,wave_period_s,wave_direction_deg,saral_sst_c,saral_distance_km,saral_source,...,rainfall_mean,rainfall_max,risk_score,risk_level,agent,data_region,region,dist_coast_m,ssha_m,saral_time
0,PFZ0001,13.170278,92.640278,7.85,1.80,8.35,236.0,29.2,315.027649,NO_NEARBY_SARAL,...,NaN,NaN,0.183336,LOW,SAFETY_AGENT,BROADER_INDIAN_MARINE_REGION,ANDAMAN,NaN,NaN,NaN
1,PFZ0002,11.760000,92.946944,7.03,0.92,6.75,203.0,29.1,309.144067,NO_NEARBY_SARAL,...,NaN,NaN,0.138278,LOW,SAFETY_AGENT,BROADER_INDIAN_MARINE_REGION,ANDAMAN,NaN,NaN,NaN
2,PFZ0003,11.745000,93.004444,7.03,1.20,7.50,207.0,28.9,302.698105,NO_NEARBY_SARAL,...,NaN,NaN,0.136969,LOW,SAFETY_AGENT,BROADER_INDIAN_MARINE_REGION,ANDAMAN,NaN,NaN,NaN
3,PFZ0004,11.694444,93.021944,7.69,1.20,7.50,207.0,28.9,299.695597,NO_NEARBY_SARAL,...,NaN,NaN,0.135210,LOW,SAFETY_AGENT,BROADER_INDIAN_MARINE_REGION,ANDAMAN,NaN,NaN,NaN
4,PFZ0005,11.632500,93.021111,8.02,1.28,7.65,212.0,28.9,298.366690,NO_NEARBY_SARAL,...,NaN,NaN,0.133388,LOW,SAFETY_AGENT,BROADER_INDIAN_MARINE_REGION,ANDAMAN,NaN,NaN,NaN


In [ ]:
def make_json_safe(obj):
    """
    Convert Pandas/NumPy objects into strict JSON-compatible
    Python objects.

    Missing numeric values become None, which becomes JSON null.
    """

    if isinstance(obj, dict):
        return {
            str(key): make_json_safe(value)
            for key, value in obj.items()
        }

    if isinstance(obj, list):
        return [
            make_json_safe(value)
            for value in obj
        ]

    if isinstance(obj, tuple):
        return [
            make_json_safe(value)
            for value in obj
        ]

    # Handle Pandas/NumPy missing values
    if pd.isna(obj):
        return None

    # NumPy scalar types
    if isinstance(obj, np.integer):
        return int(obj)

    if isinstance(obj, np.floating):
        return float(obj)

    if isinstance(obj, np.bool_):
        return bool(obj)

    return obj

In [ ]:
SAFETY_FEATURES = [
    "wind_speed_ms",
    "wave_height_m",
    "current_speed_ms",
    "cyclone_distance_km",
    "cyclone_wind_kt",
    "cyclone_grade",
    "rainfall_max"
]

print("Safety features:")
for feature in SAFETY_FEATURES:
    print(" -", feature)

Safety features:
 - wind_speed_ms
 - wave_height_m
 - current_speed_ms
 - cyclone_distance_km
 - cyclone_wind_kt
 - cyclone_grade
 - rainfall_max


In [ ]:
def wind_safety_score(wind):
    if pd.isna(wind):
        return np.nan

    return np.interp(
        wind,
        [0, 4, 6, 8, 10, 12, 15],
        [100, 100, 90, 65, 35, 10, 0]
    )

def wave_safety_score(wave):
    if pd.isna(wave):
        return np.nan

    return np.interp(
        wave,
        [0, 0.5, 1, 1.5, 2, 2.5, 3.5, 5, 8, 10],
        [100, 100, 95, 85, 70, 50, 25, 10, 0, 0]
    )

def current_safety_score(current):
    if pd.isna(current):
        return np.nan

    return np.interp(
        current,
        [0, 0.1, 0.2, 0.3, 0.4, 0.6, 0.8, 1.0],
        [100, 100, 90, 75, 55, 25, 5, 0]
    )

def rainfall_safety_score(rainfall):
    if pd.isna(rainfall):
        return np.nan

    return np.interp(
        rainfall,
        [0, 2, 4, 6, 8, 10, 15],
        [100, 95, 85, 65, 40, 20, 0]
    )

In [ ]:
CYCLONE_GRADE_SAFETY = {
    "D": 90,
    "DD": 80,
    "CS": 50,
    "SCS": 30,
    "VSCS": 15,
    "ESCS": 5
}

def cyclone_safety_score(row):
    distance = row["cyclone_distance_km"]
    wind = row["cyclone_wind_kt"]
    grade = row["cyclone_grade"]

    # If absolutely no cyclone information exists
    if pd.isna(distance) and pd.isna(wind) and pd.isna(grade):
        return np.nan

    # Distance component
    if pd.notna(distance):
        distance_score = np.interp(
            distance,
            [0, 50, 100, 200, 300, 500, 800],
            [0, 10, 30, 60, 80, 95, 100]
        )
    else:
        distance_score = 50

    # Cyclone wind component
    if pd.notna(wind):
        wind_score = np.interp(
            wind,
            [20, 30, 45, 60, 80, 100, 130],
            [100, 90, 70, 45, 20, 5, 0]
        )
    else:
        wind_score = 50

    # Cyclone grade component
    if pd.notna(grade):
        grade_score = CYCLONE_GRADE_SAFETY.get(
            str(grade).upper(),
            50
        )
    else:
        grade_score = 50

    # Combined cyclone score
    score = (
        0.45 * distance_score +
        0.40 * wind_score +
        0.15 * grade_score
    )

    return score

In [ ]:
safety_df = df.copy()

safety_df["wind_safety"] = (
    safety_df["wind_speed_ms"]
    .apply(wind_safety_score)
)

safety_df["wave_safety"] = (
    safety_df["wave_height_m"]
    .apply(wave_safety_score)
)

safety_df["current_safety"] = (
    safety_df["current_speed_ms"]
    .apply(current_safety_score)
)

safety_df["rainfall_safety"] = (
    safety_df["rainfall_max"]
    .apply(rainfall_safety_score)
)

safety_df["cyclone_safety"] = (
    safety_df.apply(cyclone_safety_score, axis=1)
)

print("Safety component scores created.")

Safety component scores created.


In [ ]:
display(
    safety_df[
        [
            "zone_id",
            "region",
            "wind_safety",
            "wave_safety",
            "cyclone_safety",
            "rainfall_safety",
            "current_safety"
        ]
    ].head(10)
)

,zone_id,region,wind_safety,wave_safety,cyclone_safety,rainfall_safety,current_safety
0,PFZ0001,ANDAMAN,66.875,76.0,77.474245,NaN,NaN
1,PFZ0002,ANDAMAN,77.125,95.8,88.937590,NaN,NaN
2,PFZ0003,ANDAMAN,77.125,91.0,89.084925,NaN,NaN
3,PFZ0004,ANDAMAN,68.875,91.0,89.282767,NaN,NaN
4,PFZ0005,ANDAMAN,64.700,89.4,89.487709,NaN,NaN
5,PFZ0006,ANDAMAN,58.700,89.4,89.645115,NaN,100.0
6,PFZ0007,ANDAMAN,59.150,90.6,89.681882,NaN,100.0
7,PFZ0008,ANDAMAN,59.150,90.6,89.738952,NaN,100.0
8,PFZ0009,ANDAMAN,51.950,90.6,89.865925,NaN,100.0
9,PFZ0010,ANDAMAN,49.100,86.2,90.331407,18.799999,100.0


In [ ]:
# Safety weights based on the ORCA project blueprint
SAFETY_WEIGHTS = {
    "wind_safety": 0.30,
    "wave_safety": 0.25,
    "cyclone_safety": 0.25,
    "rainfall_safety": 0.10,
    "current_safety": 0.10
}

def calculate_safety_score(row):

    weighted_sum = 0.0
    total_weight = 0.0

    for feature, weight in SAFETY_WEIGHTS.items():

        value = row[feature]

        if pd.notna(value):
            weighted_sum += float(value) * weight
            total_weight += weight

    if total_weight == 0:
        return np.nan

    return weighted_sum / total_weight


safety_df["safety_score"] = (
    safety_df.apply(calculate_safety_score, axis=1)
)

safety_df["safety_score"] = (
    safety_df["safety_score"].clip(0, 100)
)

print("Safety score calculated.")
print("Score range:",
      round(safety_df["safety_score"].min(), 2),
      "to",
      round(safety_df["safety_score"].max(), 2))

Safety score calculated.
Score range: 15.29 to 96.62


In [ ]:
def safety_level(score):

    if pd.isna(score):
        return "UNKNOWN"

    if score >= 80:
        return "SAFE"

    elif score >= 60:
        return "MODERATE"

    elif score >= 40:
        return "CAUTION"

    else:
        return "UNSAFE"

safety_df["safety_level"] = (
    safety_df["safety_score"].apply(safety_level)
)

In [ ]:
CONFIDENCE_FEATURES = [
    "wind_speed_ms",
    "wave_height_m",
    "current_speed_ms",
    "cyclone_distance_km",
    "cyclone_wind_kt",
    "rainfall_max"
]

safety_df["data_confidence"] = (
    safety_df[CONFIDENCE_FEATURES]
    .notna()
    .mean(axis=1)
)

safety_df["confidence_percent"] = (
    safety_df["data_confidence"] * 100
).round(1)

In [ ]:
OUTPUT_COLUMNS = [
    "zone_id",
    "region",
    "latitude",
    "longitude",
    "wind_speed_ms",
    "wave_height_m",
    "current_speed_ms",
    "cyclone_distance_km",
    "cyclone_wind_kt",
    "cyclone_grade",
    "rainfall_max",
    "wind_safety",
    "wave_safety",
    "cyclone_safety",
    "rainfall_safety",
    "current_safety",
    "safety_score",
    "safety_level",
    "confidence_percent"
]

display(
    safety_df[OUTPUT_COLUMNS].head(20)
)

,zone_id,region,latitude,longitude,wind_speed_ms,wave_height_m,current_speed_ms,cyclone_distance_km,cyclone_wind_kt,cyclone_grade,rainfall_max,wind_safety,wave_safety,cyclone_safety,rainfall_safety,current_safety,safety_score,safety_level,confidence_percent
0,PFZ0001,ANDAMAN,13.170278,92.640278,7.85,1.80,NaN,192.401813,25.0,D,NaN,66.875,76.0,77.474245,NaN,NaN,73.038826,MODERATE,66.7
1,PFZ0002,ANDAMAN,11.760000,92.946944,7.03,0.92,NaN,342.595262,25.0,D,NaN,77.125,95.8,88.937590,NaN,NaN,86.652372,SAFE,66.7
2,PFZ0003,ANDAMAN,11.745000,93.004444,7.03,1.20,NaN,346.960737,25.0,D,NaN,77.125,91.0,89.084925,NaN,NaN,85.198414,SAFE,66.7
3,PFZ0004,ANDAMAN,11.694444,93.021944,7.69,1.20,NaN,352.822719,25.0,D,NaN,68.875,91.0,89.282767,NaN,NaN,82.166490,SAFE,66.7
4,PFZ0005,ANDAMAN,11.632500,93.021111,8.02,1.28,NaN,358.895068,25.0,D,NaN,64.700,89.4,89.487709,NaN,NaN,80.164909,SAFE,66.7
5,PFZ0006,ANDAMAN,11.587778,93.025833,8.42,1.28,0.058917,363.558975,25.0,D,NaN,58.700,89.4,89.645115,NaN,100.0,80.412532,SAFE,83.3
6,PFZ0007,ANDAMAN,11.563889,92.999444,8.39,1.22,0.058917,364.648352,25.0,D,NaN,59.150,90.6,89.681882,NaN,100.0,80.906078,SAFE,83.3
7,PFZ0008,ANDAMAN,11.535556,92.975556,8.39,1.22,0.058917,366.339307,25.0,D,NaN,59.150,90.6,89.738952,NaN,100.0,80.921931,SAFE,83.3
8,PFZ0009,ANDAMAN,11.501944,92.983889,8.87,1.22,0.058917,370.101481,25.0,D,NaN,51.950,90.6,89.865925,NaN,100.0,78.557201,MODERATE,83.3
9,PFZ0010,ANDAMAN,11.362500,92.978333,9.06,1.44,0.070516,383.893532,25.0,D,10.3,49.100,86.2,90.331407,18.799999,100.0,70.742852,MODERATE,100.0


In [ ]:
def get_safest_zones(region=None, top_k=5):

    data = safety_df.copy()

    if region:
        data = data[
            data["region"].str.upper() == region.upper()
        ]

    data = data.dropna(subset=["safety_score"])

    result = (
        data
        .sort_values("safety_score", ascending=False)
        .head(top_k)
        .reset_index(drop=True)
    )

    return result

In [ ]:
get_safest_zones(top_k=10)[
    [
        "zone_id",
        "region",
        "safety_score",
        "safety_level",
        "confidence_percent"
    ]
]

,zone_id,region,safety_score,safety_level,confidence_percent
0,GRID_0702,INDIAN_MARINE,96.615851,SAFE,66.7
1,GRID_1214,INDIAN_MARINE,96.360832,SAFE,50.0
2,GRID_1135,INDIAN_MARINE,96.147289,SAFE,50.0
3,GRID_0406,INDIAN_MARINE,96.075472,SAFE,50.0
4,GRID_0452,INDIAN_MARINE,95.334806,SAFE,50.0
5,GRID_0937,INDIAN_MARINE,95.181503,SAFE,83.3
6,GRID_0797,INDIAN_MARINE,95.142854,SAFE,50.0
7,GRID_1392,INDIAN_MARINE,95.127908,SAFE,83.3
8,PFZ0071,KERALA,95.112043,SAFE,83.3
9,PFZ0070,KERALA,95.101830,SAFE,83.3


In [ ]:
kerala_safe = get_safest_zones(
    region="KERALA",
    top_k=10
)

display(
    kerala_safe[
        [
            "zone_id",
            "region",
            "safety_score",
            "safety_level",
            "confidence_percent"
        ]
    ]
)

,zone_id,region,safety_score,safety_level,confidence_percent
0,PFZ0071,KERALA,95.112043,SAFE,83.3
1,PFZ0070,KERALA,95.101830,SAFE,83.3
2,PFZ0068,KERALA,95.082538,SAFE,83.3
3,PFZ0073,KERALA,95.030724,SAFE,83.3
4,PFZ0072,KERALA,95.027187,SAFE,83.3
5,PFZ0075,KERALA,94.696608,SAFE,83.3
6,PFZ0074,KERALA,94.682441,SAFE,83.3
7,PFZ0069,KERALA,94.193347,SAFE,83.3
8,PFZ0081,KERALA,93.876475,SAFE,83.3
9,PFZ0082,KERALA,93.476552,SAFE,83.3


In [ ]:
def get_zone(zone_id):

    zone_id = zone_id.upper().strip()

    result = safety_df[
        safety_df["zone_id"].str.upper() == zone_id
    ]

    if result.empty:
        return None

    return result.iloc[0]

zone = get_zone("PFZ0074")

print(zone)

zone_id                                           PFZ0074
latitude                                        11.618889
longitude                                       75.204444
wind_speed_ms                                        4.03
wave_height_m                                        0.98
wave_period_s                                        10.5
wave_direction_deg                                  239.0
saral_sst_c                                          28.0
saral_distance_km                              116.404012
saral_source                              NO_NEARBY_SARAL
current_u_ms                                       0.0255
current_v_ms                                      -0.1295
current_speed_ms                                 0.131987
cyclone_distance_km                             466.40757
cyclone_wind_kt                                      25.0
cyclone_grade                                         NaN
cyclone_pressure_hpa                                  NaN
cyclone_pressu

In [ ]:
def zone_safety_summary(zone_id):

    zone = get_zone(zone_id)

    if zone is None:
        return {
            "status": "error",
            "message": f"Zone {zone_id.upper()} was not found."
        }

    def value_or_none(value):
        if pd.isna(value):
            return None
        return value

    def rounded_value(value, decimals=2):
        if pd.isna(value):
            return None
        return round(float(value), decimals)

    return {
        "status": "success",

        "zone_id": str(zone["zone_id"]),
        "region": str(zone["region"]),

        "latitude": rounded_value(
            zone["latitude"], 6
        ),

        "longitude": rounded_value(
            zone["longitude"], 6
        ),

        "safety_score": rounded_value(
            zone["safety_score"], 2
        ),

        "safety_level": str(
            zone["safety_level"]
        ),

        "confidence_percent": rounded_value(
            zone["confidence_percent"], 1
        ),

        "evidence": {

            "wind_speed_ms": rounded_value(
                zone["wind_speed_ms"], 2
            ),

            "wave_height_m": rounded_value(
                zone["wave_height_m"], 2
            ),

            "wave_period_s": rounded_value(
                zone["wave_period_s"], 2
            ),

            "wave_direction_deg": rounded_value(
                zone["wave_direction_deg"], 2
            ),

            "current_speed_ms": rounded_value(
                zone["current_speed_ms"], 3
            ),

            "cyclone_distance_km": rounded_value(
                zone["cyclone_distance_km"], 2
            ),

            "cyclone_wind_kt": rounded_value(
                zone["cyclone_wind_kt"], 1
            ),

            "cyclone_grade": value_or_none(
                zone["cyclone_grade"]
            ),

            "rainfall_max": rounded_value(
                zone["rainfall_max"], 2
            )
        },

        "component_scores": {

            "wind_safety": rounded_value(
                zone["wind_safety"], 2
            ),

            "wave_safety": rounded_value(
                zone["wave_safety"], 2
            ),

            "cyclone_safety": rounded_value(
                zone["cyclone_safety"], 2
            ),

            "rainfall_safety": rounded_value(
                zone["rainfall_safety"], 2
            ),

            "current_safety": rounded_value(
                zone["current_safety"], 2
            )
        }
    }

In [ ]:
result = zone_safety_summary("PFZ0074")

print(json.dumps(result, indent=2, default=str))

{
  "status": "success",
  "zone_id": "PFZ0074",
  "region": "KERALA",
  "latitude": 11.618889,
  "longitude": 75.204444,
  "safety_score": 94.68,
  "safety_level": "SAFE",
  "confidence_percent": 83.3,
  "evidence": {
    "wind_speed_ms": 4.03,
    "wave_height_m": 0.98,
    "wave_period_s": 10.5,
    "wave_direction_deg": 239.0,
    "current_speed_ms": 0.132,
    "cyclone_distance_km": 466.41,
    "cyclone_wind_kt": 25.0,
    "cyclone_grade": null,
    "rainfall_max": null
  },
  "component_scores": {
    "wind_safety": 99.85,
    "wave_safety": 95.2,
    "cyclone_safety": 87.12,
    "rainfall_safety": null,
    "current_safety": 96.8
  }
}


In [ ]:
def compare_zones(zone1_id, zone2_id):

    zone1 = get_zone(zone1_id)
    zone2 = get_zone(zone2_id)

    if zone1 is None:
        return {
            "status": "error",
            "message": f"{zone1_id} was not found."
        }

    if zone2 is None:
        return {
            "status": "error",
            "message": f"{zone2_id} was not found."
        }

    score1 = zone1["safety_score"]
    score2 = zone2["safety_score"]

    if score1 > score2:
        safer = zone1_id.upper()

    elif score2 > score1:
        safer = zone2_id.upper()

    else:
        safer = "EQUAL"

    return {
        "status": "success",

        "zone_1": {
            "zone_id": zone1_id.upper(),
            "region": zone1["region"],
            "safety_score": round(float(score1), 2),
            "safety_level": zone1["safety_level"]
        },

        "zone_2": {
            "zone_id": zone2_id.upper(),
            "region": zone2["region"],
            "safety_score": round(float(score2), 2),
            "safety_level": zone2["safety_level"]
        },

        "safer_zone": safer
    }

print(
    json.dumps(
        compare_zones("PFZ0070", "PFZ0071"),
        indent=2
    )
)

{
  "status": "success",
  "zone_1": {
    "zone_id": "PFZ0070",
    "region": "KERALA",
    "safety_score": 95.1,
    "safety_level": "SAFE"
  },
  "zone_2": {
    "zone_id": "PFZ0071",
    "region": "KERALA",
    "safety_score": 95.11,
    "safety_level": "SAFE"
  },
  "safer_zone": "PFZ0071"
}


In [ ]:
def get_unsafe_zones(
    region=None,
    threshold=40,
    top_k=10
):

    data = safety_df.copy()

    if region:
        data = data[
            data["region"].str.upper() == region.upper()
        ]

    result = data[
        data["safety_score"] < threshold
    ].sort_values(
        "safety_score",
        ascending=True
    ).head(top_k)

    return result.reset_index(drop=True)

display(
    get_unsafe_zones(top_k=10)[
        [
            "zone_id",
            "region",
            "safety_score",
            "safety_level",
            "cyclone_distance_km",
            "cyclone_wind_kt",
            "wave_height_m",
            "wind_speed_ms"
        ]
    ]
)

,zone_id,region,safety_score,safety_level,cyclone_distance_km,cyclone_wind_kt,wave_height_m,wind_speed_ms
0,PFZ0334,SOUTH ANDHRAPRADESH,15.286965,UNSAFE,23.330478,55.0,9.378,11.58
1,GRID_1356,INDIAN_MARINE,21.164737,UNSAFE,114.183236,90.0,NaN,NaN
2,PFZ0335,SOUTH ANDHRAPRADESH,27.304464,UNSAFE,33.842427,55.0,9.772,10.77
3,GRID_1652,INDIAN_MARINE,32.261698,UNSAFE,60.713135,55.0,NaN,NaN
4,GRID_1234,INDIAN_MARINE,35.576088,UNSAFE,60.607897,50.0,NaN,NaN
5,GRID_0881,INDIAN_MARINE,35.889982,UNSAFE,60.660966,55.0,6.404,9.35
6,GRID_1649,INDIAN_MARINE,36.741718,UNSAFE,85.602137,55.0,NaN,NaN
7,GRID_0697,INDIAN_MARINE,37.106858,UNSAFE,222.389853,25.0,NaN,13.22


In [ ]:
def get_cyclone_risk_zones(
    max_distance_km=200,
    top_k=10
):

    data = safety_df[
        safety_df["cyclone_distance_km"] <= max_distance_km
    ].copy()

    data = data.sort_values(
        [
            "cyclone_distance_km",
            "cyclone_wind_kt"
        ],
        ascending=[
            True,
            False
        ]
    )

    return data.head(top_k).reset_index(drop=True)

display(
    get_cyclone_risk_zones()[
        [
            "zone_id",
            "region",
            "cyclone_distance_km",
            "cyclone_wind_kt",
            "cyclone_grade",
            "safety_score"
        ]
    ]
)

,zone_id,region,cyclone_distance_km,cyclone_wind_kt,cyclone_grade,safety_score
0,GRID_1106,INDIAN_MARINE,0.0,55.0,SCS,56.577004
1,GRID_1213,INDIAN_MARINE,0.0,55.0,SCS,60.885218
2,GRID_1616,INDIAN_MARINE,0.0,55.0,SCS,75.353704
3,GRID_1531,INDIAN_MARINE,0.0,45.0,CS,68.375940
4,GRID_1139,INDIAN_MARINE,0.0,40.0,CS,77.158333
5,GRID_1343,INDIAN_MARINE,0.0,40.0,CS,58.375573
6,GRID_1223,INDIAN_MARINE,0.0,35.0,CS,67.370833
7,GRID_1402,INDIAN_MARINE,0.0,35.0,CS,56.763333
8,GRID_0241,INDIAN_MARINE,0.0,30.0,DD,63.934630
9,GRID_0792,INDIAN_MARINE,0.0,30.0,DD,55.380000


In [ ]:
REGIONS = [
    "KERALA",
    "MAHARASHTRA",
    "NORTH TAMILNADU",
    "SOUTH ANDHRAPRADESH",
    "ANDAMAN",
    "KARNATAKA",
    "LAKSHADWEEP",
    "GOA",
    "INDIAN_MARINE"
]

def detect_region(query):

    query = query.lower()

    for region in REGIONS:

        normalized = region.lower().replace("_", " ")

        if normalized in query:
            return region

    return None

def extract_zone_ids(query):

    query = query.upper()

    matches = re.findall(
        r'\b(?:PFZ\d{4}|GRID_\d{4})\b',
        query
    )

    return list(dict.fromkeys(matches))

print(
    extract_zone_ids(
        "Compare PFZ0074 and PFZ0075"
    )
)

['PFZ0074', 'PFZ0075']


In [ ]:
def parse_safety_query(query):

    q = query.lower().strip()

    zones = extract_zone_ids(query)
    region = detect_region(query)

    # ==================================================
    # 1. TEMPORAL / FORECAST QUERIES
    # ==================================================

    temporal_keywords = [
        "tomorrow",
        "after 9 am",
        "after 9am",
        "after 10 am",
        "after 10am",
        "morning",
        "afternoon",
        "evening",
        "later today",
        "remain safe",
        "will it remain",
        "what time"
    ]

    if any(keyword in q for keyword in temporal_keywords):

        return {
            "query_type": "forecast_not_supported",
            "zone_id": zones[0] if zones else None,
            "zone1": None,
            "zone2": None,
            "region": region,
            "top_k": 5
        }

    # ==================================================
    # 2. COMPARE ZONES
    # ==================================================

    if (
        (
            "compare" in q
            or "which is safer" in q
            or "safer" in q
        )
        and len(zones) >= 2
    ):

        return {
            "query_type": "compare_zones",
            "zone1": zones[0],
            "zone2": zones[1],
            "zone_id": None,
            "region": region,
            "top_k": 5
        }

    # ==================================================
    # 3. SPECIFIC ZONE
    # ==================================================

    if zones:

        if (
            "cyclone" in q
            or "storm" in q
        ):
            query_type = "zone_cyclone"

        elif "wave" in q:

            query_type = "zone_wave"

        elif "wind" in q:

            query_type = "zone_wind"

        else:

            query_type = "zone_safety"

        return {
            "query_type": query_type,
            "zone_id": zones[0],
            "zone1": None,
            "zone2": None,
            "region": region,
            "top_k": 5
        }

    # ==================================================
    # 4. UNSAFE / AVOID
    # ==================================================

    if (
        "avoid" in q
        or "unsafe" in q
        or "dangerous zones" in q
    ):

        return {
            "query_type": "unsafe_zones",
            "zone_id": None,
            "zone1": None,
            "zone2": None,
            "region": region,
            "top_k": 5
        }

    # ==================================================
    # 5. CYCLONE RISK
    # ==================================================

    if (
        "cyclone" in q
        or "storm" in q
    ):

        return {
            "query_type": "cyclone_risk",
            "zone_id": None,
            "zone1": None,
            "zone2": None,
            "region": region,
            "top_k": 5
        }

    # ==================================================
    # 6. SAFEST ZONES
    # ==================================================

    if (
        "safest" in q
        or "safe zones" in q
        or "best safe" in q
        or "which zones are safe" in q
    ):

        return {
            "query_type": "safe_zones",
            "zone_id": None,
            "zone1": None,
            "zone2": None,
            "region": region,
            "top_k": 5
        }

    # ==================================================
    # 7. GENERAL SAFETY
    # ==================================================

    if (
        "safe" in q
        or "safety" in q
        or "risk" in q
        or "conditions" in q
    ):

        return {
            "query_type": "general_safety",
            "zone_id": None,
            "zone1": None,
            "zone2": None,
            "region": region,
            "top_k": 5
        }

    # ==================================================
    # 8. UNKNOWN
    # ==================================================

    return {
        "query_type": "unknown",
        "zone_id": None,
        "zone1": None,
        "zone2": None,
        "region": region,
        "top_k": 5
    }

In [ ]:
def safety_agent(query):

    parsed = parse_safety_query(query)

    query_type = parsed["query_type"]

    # ================================================
    # SPECIFIC ZONE
    # ================================================

    if query_type in [
        "zone_safety",
        "zone_wave",
        "zone_wind",
        "zone_cyclone"
    ]:

        result = zone_safety_summary(
            parsed["zone_id"]
        )

        return {
            "agent": "safety_agent",
            "status": "success",
            "query_type": query_type,
            "request": parsed,
            "data": [result]
        }

    # ================================================
    # COMPARE
    # ================================================

    elif query_type == "compare_zones":

        result = compare_zones(
            parsed["zone1"],
            parsed["zone2"]
        )

        return {
            "agent": "safety_agent",
            "status": "success",
            "query_type": query_type,
            "request": parsed,
            "data": [result]
        }

    # ================================================
    # SAFEST ZONES
    # ================================================

    elif query_type == "safe_zones":

        result = get_safest_zones(
            region=parsed["region"],
            top_k=parsed["top_k"]
        )

        return {
            "agent": "safety_agent",
            "status": "success",
            "query_type": query_type,
            "request": parsed,
            "data": result.to_dict(
                orient="records"
            )
        }

    # ================================================
    # UNSAFE ZONES
    # ================================================

    elif query_type == "unsafe_zones":

        result = get_unsafe_zones(
            region=parsed["region"],
            top_k=parsed["top_k"]
        )

        return {
            "agent": "safety_agent",
            "status": "success",
            "query_type": query_type,
            "request": parsed,
            "data": result.to_dict(
                orient="records"
            )
        }

    # ================================================
    # CYCLONE
    # ================================================

    elif query_type == "cyclone_risk":

        result = get_cyclone_risk_zones(
            top_k=parsed["top_k"]
        )

        return {
            "agent": "safety_agent",
            "status": "success",
            "query_type": query_type,
            "request": parsed,
            "data": result.to_dict(
                orient="records"
            )
        }

    # ================================================
    # TEMPORAL
    # ================================================

    elif query_type == "forecast_not_supported":

        return {
            "agent": "safety_agent",
            "status": "not_supported",
            "query_type": "forecast_not_supported",
            "request": parsed,
            "data": [],
            "message": (
                "Forecast-based safety queries are not supported "
                "by the current dataset. Forecast data will be "
                "integrated in a later version."
            )
        }

    # ================================================
    # UNKNOWN
    # ================================================

    else:

        return {
            "agent": "safety_agent",
            "status": "error",
            "query_type": "unknown",
            "request": parsed,
            "data": [],
            "message": (
                "I could not determine the safety query."
            )
        }

In [ ]:
def generate_safety_answer(agent_result):

    query_type = agent_result["query_type"]

    # ================================================
    # SPECIFIC ZONE
    # ================================================

    if query_type in [
        "zone_safety",
        "zone_wave",
        "zone_wind",
        "zone_cyclone"
    ]:

        if not agent_result["data"]:
            return "The requested zone could not be found."

        data = agent_result["data"][0]

        if data.get("status") == "error":
            return data["message"]

        zone = data["zone_id"]
        region = data["region"]

        score = data["safety_score"]
        level = data["safety_level"]
        confidence = data["confidence_percent"]

        # Environmental measurements are stored
        # inside the evidence dictionary.
        evidence = data.get("evidence", {})

        # ============================================
        # OVERALL SAFETY
        # ============================================

        if query_type == "zone_safety":

            answer = (
                f"{zone} in {region} has a safety score "
                f"of {score:.2f}/100 ({level})."
            )

            wind = evidence.get("wind_speed_ms")
            wave = evidence.get("wave_height_m")
            cyclone_distance = evidence.get(
                "cyclone_distance_km"
            )

            if wind is not None:
                answer += (
                    f" Wind speed is "
                    f"{wind:.2f} m/s."
                )

            if wave is not None:
                answer += (
                    f" Wave height is "
                    f"{wave:.2f} m."
                )

            if cyclone_distance is not None:
                answer += (
                    f" The nearest cyclone-related "
                    f"distance in the dataset is "
                    f"{cyclone_distance:.1f} km."
                )

            answer += (
                f" Data confidence is "
                f"{confidence:.1f}%."
            )

            return answer

        # ============================================
        # WAVE CONDITIONS
        # ============================================

        if query_type == "zone_wave":

            wave = evidence.get("wave_height_m")
            period = evidence.get("wave_period_s")
            direction = evidence.get(
                "wave_direction_deg"
            )

            if wave is None:
                return (
                    f"Wave data is unavailable for "
                    f"{zone}."
                )

            answer = (
                f"{zone} has a wave height of "
                f"{wave:.2f} m."
            )

            if period is not None:
                answer += (
                    f" Wave period is "
                    f"{period:.2f} s."
                )

            if direction is not None:
                answer += (
                    f" Wave direction is "
                    f"{direction:.1f}°."
                )

            answer += (
                f" The overall safety score is "
                f"{score:.2f}/100 ({level})."
            )

            answer += (
                f" Data confidence is "
                f"{confidence:.1f}%."
            )

            return answer

        # ============================================
        # WIND CONDITIONS
        # ============================================

        if query_type == "zone_wind":

            wind = evidence.get("wind_speed_ms")

            if wind is None:
                return (
                    f"Wind data is unavailable for "
                    f"{zone}."
                )

            component_scores = data.get(
                "component_scores", {}
            )

            wind_safety = component_scores.get(
                "wind_safety"
            )

            answer = (
                f"{zone} has a wind speed of "
                f"{wind:.2f} m/s."
            )

            if wind_safety is not None:
                answer += (
                    f" The wind safety component "
                    f"score is "
                    f"{wind_safety:.2f}/100."
                )

            answer += (
                f" The overall safety score is "
                f"{score:.2f}/100 ({level})."
            )

            answer += (
                f" Data confidence is "
                f"{confidence:.1f}%."
            )

            return answer

        # ============================================
        # CYCLONE / STORM RISK
        # ============================================

        if query_type == "zone_cyclone":

            distance = evidence.get(
                "cyclone_distance_km"
            )

            cyclone_wind = evidence.get(
                "cyclone_wind_kt"
            )

            cyclone_grade = evidence.get(
                "cyclone_grade"
            )

            if distance is None:
                return (
                    f"Cyclone-related data is "
                    f"unavailable for {zone}."
                )

            component_scores = data.get(
                "component_scores", {}
            )

            cyclone_safety = component_scores.get(
                "cyclone_safety"
            )

            answer = (
                f"For {zone}, the nearest "
                f"cyclone-related distance in the "
                f"dataset is "
                f"{distance:.1f} km."
            )

            if cyclone_wind is not None:
                answer += (
                    f" Cyclone wind is "
                    f"{cyclone_wind:.0f} kt."
                )

            if cyclone_grade is not None:
                answer += (
                    f" Cyclone grade is "
                    f"{cyclone_grade}."
                )

            if cyclone_safety is not None:
                answer += (
                    f" The cyclone safety component "
                    f"score is "
                    f"{cyclone_safety:.2f}/100."
                )

            answer += (
                f" The overall safety score is "
                f"{score:.2f}/100 ({level})."
            )

            answer += (
                f" Data confidence is "
                f"{confidence:.1f}%."
            )

            return answer

    # ================================================
    # COMPARE
    # ================================================

    if query_type == "compare_zones":

        if not agent_result["data"]:
            return "The requested zones could not be compared."

        data = agent_result["data"][0]

        z1 = data["zone_1"]
        z2 = data["zone_2"]

        if data["safer_zone"] == "EQUAL":

            return (
                f"{z1['zone_id']} and {z2['zone_id']} "
                f"have equal safety scores of "
                f"{z1['safety_score']:.2f}/100."
            )

        safer = data["safer_zone"]

        return (
            f"{safer} is safer based on the available "
            f"environmental conditions.\n\n"
            f"{z1['zone_id']}: "
            f"{z1['safety_score']:.2f}/100 "
            f"({z1['safety_level']})\n"
            f"{z2['zone_id']}: "
            f"{z2['safety_score']:.2f}/100 "
            f"({z2['safety_level']})"
        )

    # ================================================
    # SAFEST ZONES
    # ================================================

    if query_type == "safe_zones":

        data = agent_result["data"]

        if not data:
            return "No suitable safe zones were found."

        region = agent_result["request"].get("region")

        title = (
            f"The safest zones in {region} are:"
            if region
            else
            "The safest zones in the available dataset are:"
        )

        lines = []

        for i, item in enumerate(data, 1):

            lines.append(
                f"{i}. {item['zone_id']} "
                f"— {item['safety_score']:.2f}/100 "
                f"({item['safety_level']})"
            )

        return title + "\n" + "\n".join(lines)

    # ================================================
    # UNSAFE ZONES
    # ================================================

    if query_type == "unsafe_zones":

        data = agent_result["data"]

        if not data:
            return (
                "No zones below the unsafe threshold "
                "were found."
            )

        lines = []

        for item in data:

            lines.append(
                f"{item['zone_id']} — "
                f"{item['safety_score']:.2f}/100 "
                f"({item['safety_level']})"
            )

        return (
            "Zones with the lowest safety scores are:\n"
            + "\n".join(lines)
        )

    # ================================================
    # CYCLONE RISK ACROSS ZONES
    # ================================================

    if query_type == "cyclone_risk":

        data = agent_result["data"]

        if not data:
            return (
                "No nearby cyclone-related zones "
                "were found."
            )

        lines = []

        for item in data:

            lines.append(
                f"{item['zone_id']}: cyclone distance "
                f"{item['cyclone_distance_km']:.1f} km, "
                f"cyclone wind "
                f"{item['cyclone_wind_kt']:.0f} kt, "
                f"safety score "
                f"{item['safety_score']:.2f}"
            )

        return (
            "Zones with the strongest nearby cyclone "
            "signals in the available data:\n"
            + "\n".join(lines)
        )

    # ================================================
    # TEMPORAL / FORECAST
    # ================================================

    if query_type == "forecast_not_supported":

        return agent_result["message"]

    # ================================================
    # FALLBACK
    # ================================================

    return agent_result.get(
        "message",
        "Unable to generate a safety response."
    )

In [ ]:
def ask_safety_agent(query):

    result = safety_agent(query)

    result["answer"] = generate_safety_answer(result)

    return make_json_safe(result)

result = ask_safety_agent(
    "Is PFZ0074 safe for my boat?"
)

print(result["answer"])

PFZ0074 in KERALA has a safety score of 94.68/100 (SAFE). Wind speed is 4.03 m/s. Wave height is 0.98 m. The nearest cyclone-related distance in the dataset is 466.4 km. Data confidence is 83.3%.


In [ ]:
test_queries = [

    "Is PFZ0074 safe for my boat?",

    "Which is safer, PFZ0070 or PFZ0071?",

    "Which fishing zones are safest in Kerala?",

    "What is the current safety score of PFZ0074?",

    "What are the wave conditions at PFZ0074?",

    "Are the wind conditions safe for PFZ0074?",

    "Is there any cyclone or storm risk near PFZ0074?",

    "Which zones should I avoid?",

    "Which zones should I avoid in Kerala?",

    "What are the safest zones in Kerala?",

    "What are the current safest fishing zones?",

    "Is PFZ0074 safe?",

    "What is the safety level at PFZ0074?",

    "What time tomorrow is safest to go fishing?",

    "Will PFZ0074 remain safe after 9 AM?"
]

for i, query in enumerate(test_queries, 1):

    print("\n" + "=" * 80)
    print(f"TEST {i}")
    print("QUERY:", query)

    result = ask_safety_agent(query)

    print("TYPE:", result["query_type"])
    print("STATUS:", result["status"])
    print("ANSWER:")
    print(result["answer"])


TEST 1
QUERY: Is PFZ0074 safe for my boat?
TYPE: zone_safety
STATUS: success
ANSWER:
PFZ0074 in KERALA has a safety score of 94.68/100 (SAFE). Wind speed is 4.03 m/s. Wave height is 0.98 m. The nearest cyclone-related distance in the dataset is 466.4 km. Data confidence is 83.3%.

TEST 2
QUERY: Which is safer, PFZ0070 or PFZ0071?
TYPE: compare_zones
STATUS: success
ANSWER:
PFZ0071 is safer based on the available environmental conditions.

PFZ0070: 95.10/100 (SAFE)
PFZ0071: 95.11/100 (SAFE)

TEST 3
QUERY: Which fishing zones are safest in Kerala?
TYPE: safe_zones
STATUS: success
ANSWER:
The safest zones in KERALA are:
1. PFZ0071 — 95.11/100 (SAFE)
2. PFZ0070 — 95.10/100 (SAFE)
3. PFZ0068 — 95.08/100 (SAFE)
4. PFZ0073 — 95.03/100 (SAFE)
5. PFZ0072 — 95.03/100 (SAFE)

TEST 4
QUERY: What is the current safety score of PFZ0074?
TYPE: zone_safety
STATUS: success
ANSWER:
PFZ0074 in KERALA has a safety score of 94.68/100 (SAFE). Wind speed is 4.03 m/s. Wave height is 0.98 m. The nearest cyclo

In [ ]:
for region in REGIONS:

    result = get_safest_zones(
        region=region,
        top_k=3
    )

    print("\n" + "=" * 60)
    print(region)

    if result.empty:
        print("No zones found.")
        continue

    print(
        result[
            [
                "zone_id",
                "safety_score",
                "safety_level"
            ]
        ].to_string(index=False)
    )


KERALA
zone_id  safety_score safety_level
PFZ0071     95.112043         SAFE
PFZ0070     95.101830         SAFE
PFZ0068     95.082538         SAFE

MAHARASHTRA
zone_id  safety_score safety_level
PFZ0245     89.792353         SAFE
PFZ0246     89.765934         SAFE
PFZ0247     89.372183         SAFE

NORTH TAMILNADU
zone_id  safety_score safety_level
PFZ0305     86.001122         SAFE
PFZ0306     85.404112         SAFE
PFZ0309     85.197599         SAFE

SOUTH ANDHRAPRADESH
zone_id  safety_score safety_level
PFZ0315     84.081491         SAFE
PFZ0317     82.982567         SAFE
PFZ0327     79.743565     MODERATE

ANDAMAN
zone_id  safety_score safety_level
PFZ0016     92.050158         SAFE
PFZ0017     91.070773         SAFE
PFZ0014     89.542655         SAFE

KARNATAKA
zone_id  safety_score safety_level
PFZ0032     89.429172         SAFE
PFZ0034     89.328476         SAFE
PFZ0033     88.811602         SAFE

LAKSHADWEEP
zone_id  safety_score safety_level
PFZ0167     87.672538         SAF

In [ ]:
print(
    safety_df["safety_score"].describe()
)

print(
    safety_df["safety_level"].value_counts()
)

count    2037.000000
mean       76.902550
std        10.647508
min        15.286965
25%        69.925855
50%        78.796767
75%        85.093703
max        96.615851
Name: safety_score, dtype: float64
safety_level
MODERATE    973
SAFE        918
CAUTION     138
UNSAFE        8
Name: count, dtype: int64


In [ ]:
FINAL_COLUMNS = [
    "zone_id",
    "region",
    "latitude",
    "longitude",

    "wind_speed_ms",
    "wave_height_m",
    "wave_period_s",
    "wave_direction_deg",

    "current_speed_ms",

    "cyclone_distance_km",
    "cyclone_wind_kt",
    "cyclone_grade",
    "cyclone_pressure_hpa",
    "cyclone_pressure_drop_hpa",

    "rainfall_mean",
    "rainfall_max",

    "wind_safety",
    "wave_safety",
    "cyclone_safety",
    "rainfall_safety",
    "current_safety",

    "safety_score",
    "safety_level",
    "confidence_percent"
]

final_safety_df = safety_df[FINAL_COLUMNS].copy()

final_safety_df.to_csv(
    "/content/safety_scored_dataset.csv",
    index=False
)

print("Saved successfully.")

Saved successfully.


In [ ]:
print("Total rows:", len(safety_df))

print("NaN safety scores:",
      safety_df["safety_score"].isna().sum())

print("NaN safety levels:",
      safety_df["safety_level"].isna().sum())

print("NaN confidence:",
      safety_df["confidence_percent"].isna().sum())

Total rows: 2037
NaN safety scores: 0
NaN safety levels: 0
NaN confidence: 0


In [ ]:
print("===== SAFETY DATA VALIDATION =====")

print("Rows:", len(safety_df))
print("Columns:", len(safety_df.columns))

print("\nSafety score range:")
print(safety_df["safety_score"].min(),
      "to",
      safety_df["safety_score"].max())

print("\nNaN safety scores:",
      safety_df["safety_score"].isna().sum())

print("NaN safety levels:",
      safety_df["safety_level"].isna().sum())

print("NaN confidence:",
      safety_df["confidence_percent"].isna().sum())

print("\nSafety levels:")
print(safety_df["safety_level"].value_counts())

===== SAFETY DATA VALIDATION =====
Rows: 2037
Columns: 38

Safety score range:
15.286965431495974 to 96.61585077220712

NaN safety scores: 0
NaN safety levels: 0
NaN confidence: 0

Safety levels:
safety_level
MODERATE    973
SAFE        918
CAUTION     138
UNSAFE        8
Name: count, dtype: int64


In [ ]:
CONFIDENCE_FEATURES = [
    "wind_speed_ms",
    "wave_height_m",
    "current_speed_ms",
    "cyclone_distance_km",
    "cyclone_wind_kt",
    "rainfall_max"
]

safety_df["data_confidence"] = (
    safety_df[CONFIDENCE_FEATURES]
    .notna()
    .mean(axis=1)
)

safety_df["confidence_percent"] = (
    safety_df["data_confidence"] * 100
).round(1)

print(
    "NaN confidence values:",
    safety_df["confidence_percent"].isna().sum()
)

NaN confidence values: 0


In [ ]:
display(
    safety_df[
        [
            "zone_id",
            "region",
            "wind_speed_ms",
            "wave_height_m",
            "current_speed_ms",
            "cyclone_distance_km",
            "cyclone_wind_kt",
            "rainfall_max",
            "safety_score",
            "safety_level",
            "confidence_percent"
        ]
    ]
    .sort_values("safety_score", ascending=False)
    .head(10)
)

,zone_id,region,wind_speed_ms,wave_height_m,current_speed_ms,cyclone_distance_km,cyclone_wind_kt,rainfall_max,safety_score,safety_level,confidence_percent
1038,GRID_0702,INDIAN_MARINE,3.11,NaN,0.091788,409.665541,25.0,NaN,96.615851,SAFE,66.7
1550,GRID_1214,INDIAN_MARINE,3.94,NaN,NaN,433.150530,25.0,NaN,96.360832,SAFE,50.0
1471,GRID_1135,INDIAN_MARINE,4.03,NaN,NaN,365.304791,20.0,NaN,96.147289,SAFE,50.0
742,GRID_0406,INDIAN_MARINE,3.00,NaN,NaN,414.549268,25.0,NaN,96.075472,SAFE,50.0
788,GRID_0452,INDIAN_MARINE,3.43,NaN,NaN,307.009597,20.0,NaN,95.334806,SAFE,50.0
1273,GRID_0937,INDIAN_MARINE,3.77,NaN,0.134629,443.101422,25.0,3.7,95.181503,SAFE,83.3
1133,GRID_0797,INDIAN_MARINE,4.33,NaN,NaN,412.423068,25.0,NaN,95.142854,SAFE,50.0
1728,GRID_1392,INDIAN_MARINE,3.35,1.28,0.037336,464.754597,25.0,NaN,95.127908,SAFE,83.3
70,PFZ0071,KERALA,4.41,0.72,0.069147,464.840194,25.0,NaN,95.112043,SAFE,83.3
69,PFZ0070,KERALA,4.41,0.72,0.069147,463.750752,25.0,NaN,95.101830,SAFE,83.3


In [ ]:
display(
    safety_df[
        [
            "zone_id",
            "region",
            "wind_speed_ms",
            "wave_height_m",
            "current_speed_ms",
            "cyclone_distance_km",
            "cyclone_wind_kt",
            "rainfall_max",
            "safety_score",
            "safety_level",
            "confidence_percent"
        ]
    ]
    .sort_values("safety_score", ascending=True)
    .head(10)
)

,zone_id,region,wind_speed_ms,wave_height_m,current_speed_ms,cyclone_distance_km,cyclone_wind_kt,rainfall_max,safety_score,safety_level,confidence_percent
333,PFZ0334,SOUTH ANDHRAPRADESH,11.58,9.378,NaN,23.330478,55.0,9.8,15.286965,UNSAFE,83.3
1692,GRID_1356,INDIAN_MARINE,NaN,NaN,NaN,114.183236,90.0,NaN,21.164737,UNSAFE,33.3
334,PFZ0335,SOUTH ANDHRAPRADESH,10.77,9.772,0.415188,33.842427,55.0,5.3,27.304464,UNSAFE,100.0
1988,GRID_1652,INDIAN_MARINE,NaN,NaN,NaN,60.713135,55.0,NaN,32.261698,UNSAFE,33.3
1570,GRID_1234,INDIAN_MARINE,NaN,NaN,NaN,60.607897,50.0,NaN,35.576088,UNSAFE,33.3
1217,GRID_0881,INDIAN_MARINE,9.35,6.404,0.435206,60.660966,55.0,4.4,35.889982,UNSAFE,100.0
1985,GRID_1649,INDIAN_MARINE,NaN,NaN,NaN,85.602137,55.0,NaN,36.741718,UNSAFE,33.3
1033,GRID_0697,INDIAN_MARINE,13.22,NaN,NaN,222.389853,25.0,NaN,37.106858,UNSAFE,50.0
1037,GRID_0701,INDIAN_MARINE,10.77,9.772,0.176907,107.891882,25.0,5.3,40.559776,CAUTION,100.0
949,GRID_0613,INDIAN_MARINE,10.50,9.496,0.238244,745.680481,NaN,4.1,43.474489,CAUTION,83.3


In [ ]:
import json

# Final requested test list
test_queries = [
    "Is PFZ0074 safe for my boat?",
    "Which is safer, PFZ0070 or PFZ0071?",
    "Which fishing zones are safest in Kerala?",
    "What are the wave conditions at PFZ0074?",
    "Are the wind conditions safe for PFZ0074?",
    "Is there any cyclone or storm risk near PFZ0074?",
    "Which zones should I avoid?",
    "What time tomorrow is safest to go fishing?",
    "Will PFZ0074 remain safe after 9 AM?"
]

print("===== RUNNING FINAL VALIDATION TESTS =====")
for query in test_queries:
    print(f"\n>>> Query: {query}")
    result = ask_safety_agent(query)
    # Using allow_nan=False as requested to verify make_json_safe worked
    print(json.dumps(result, indent=2, allow_nan=False))

===== RUNNING FINAL VALIDATION TESTS =====

>>> Query: Is PFZ0074 safe for my boat?
{
  "agent": "safety_agent",
  "status": "success",
  "query_type": "zone_safety",
  "request": {
    "query_type": "zone_safety",
    "zone_id": "PFZ0074",
    "zone1": null,
    "zone2": null,
    "region": null,
    "top_k": 5
  },
  "data": [
    {
      "status": "success",
      "zone_id": "PFZ0074",
      "region": "KERALA",
      "latitude": 11.618889,
      "longitude": 75.204444,
      "safety_score": 94.68,
      "safety_level": "SAFE",
      "confidence_percent": 83.3,
      "evidence": {
        "wind_speed_ms": 4.03,
        "wave_height_m": 0.98,
        "wave_period_s": 10.5,
        "wave_direction_deg": 239.0,
        "current_speed_ms": 0.132,
        "cyclone_distance_km": 466.41,
        "cyclone_wind_kt": 25.0,
        "cyclone_grade": null,
        "rainfall_max": null
      },
      "component_scores": {
        "wind_safety": 99.85,
        "wave_safety": 95.2,
        "cyclone_